#Basic Tasks

1

![](/Workspace/Users/rinkimehrasalesforce@gmail.com/Assignments/Day 11/image1.jpeg)

In [0]:
%sql
--2
CREATE CONNECTION neon_db_connection
TYPE postgresql
OPTIONS (
  host 'ep-flat-glade-atlhyv2p-pooler.c-9.us-east-1.aws.neon.tech',
  port '5432',
  user '<user_id>',
  password '<password>'
);

In [0]:
%sql
CREATE FOREIGN CATALOG neon_foreign_catalog
USING CONNECTION neon_db_connection
OPTIONS (database 'neondb');

In [0]:
%sql
select * from neon_foreign_catalog.public.doctors

3

Lakebase is a managed, Postgres-style database for apps to fast inserts, updates, deletes, one record at a time. Delta tables are for analytics big batch reads and reporting, not quick individual writes.

Building an app that needs fast, live reads/writes → use Lakebase. Doing reporting/analysis on lots of data → use Delta table. 
Lakebase can also sync into Delta automatically, so app data still flows into analytics without extra work.

#Intermediate Tasks

4

Data in Power BI can become stale in two main ways:

Scheduled Refresh: Power BI pulls in and stores a snapshot of the data at refresh time. If a scheduled refresh gets skipped or fails, the report just keeps displaying that old snapshot, with no warning — it'll look completely normal, but the numbers underneath are frozen until the next successful refresh runs.

Live Query: Instead of storing a copy, Power BI sends a fresh query to the source every time someone opens the report. This mostly avoids staleness, but it isn't foolproof — if the underlying source hasn't been updated yet, or if there's a network hiccup or slow warehouse response, the report can still end up showing outdated or incomplete results, even though it's technically asking for "live" data each time.

In [0]:
%sql
--5
select
  d.name as c_name,
  n.first_name as d_name
from cyntexa_dev.sales.customers1 d
cross join neon_foreign_catalog.public.doctors n
limit 5;

In [0]:
%sql
--6
create share gold_total_revenue_share
comment 'Sharing gold sales total revenue table with partner';

alter share gold_total_revenue_share
add table cyntexa_dev.sales.gold_total_revenue;

-- Step 1 — create the recipient
create recipient partner_recipient
using id '<recipient-sharing-identifier>';

-- Step 2 — grant access
grant select on share gold_total_revenue_share to recipient partner_recipient;

-- Step 3 — verify the share was created correctly
show shares;
show grants on share gold_total_revenue_share;

7

![](/Workspace/Users/rinkimehrasalesforce@gmail.com/Assignments/Day 11/image2.jpeg)

# Advanced Tasks

8

If the partner doesn't have a Databricks workspace → use Databricks-to-Open. They can only get tables, accessed with a token, no Databricks account needed.

If the partner has a Databricks workspace and only needs tables → either protocol works, but Databricks-to-Databricks gives better governance.

If the partner needs anything beyond tables (notebooks, models, volumes) → must use Databricks-to-Databricks, since Databricks-to-Open only supports plain Delta tables, nothing else.

9

Federation makes sense when:

data needs to be fresh/live (no acceptable delay)
query volume is low — just occasional lookups or joins, not heavy repeated scanning
the dataset is small to moderate, so live round-trips to Postgres stay fast

Federation stops making sense when:

query volume gets high — every query hits Postgres directly, so heavy/repeated queries slow down the source database and can affect the app relying on it
queries are complex or run often — repeated joins/aggregations over federation are slower than querying local Delta data, since there's network + remote execution overhead each time
freshness needs are actually loose — if daily data is good enough, paying the performance cost of live queries isn't worth it

10

- OLTP side (Lakebase):

App writes here directly — every inventory update, stock check, order placed
Fast inserts/updates/deletes, low latency, handles concurrent app traffic
Owned operationally by the application/engineering team — they manage schema, app logic, and uptime for live writes

- OLAP side (Lakehouse / Delta tables):

Used for reporting, dashboards, trend analysis (e.g. "which products are running low across all stores")
Not written to directly by the app — receives data through sync
Owned operationally by the data engineering/analytics team — they manage the gold tables, dashboards, and downstream pipelines


Lakebase automatically syncs operational changes into managed Delta tables (via its built-in change data feed)
No manual ETL job needed just to move data — sync happens continuously/near real-time
Delta tables then feed into silver/gold tables, dashboards, and Genie for business-facing analytics